In [48]:
import sys

import numpy as np
import pandas as pd
sys.path.append("/cs/casmip/alina.ryabtsev/FewShotLearning/Lemonade")
from Lemonade.VirtualRichard import VirtualRichard
from glob import glob
from Lemonade.constants import *
import os
import nibabel as nib
from Lemonade.utils import postprocess_predictions
import re

In [2]:
%reload_ext autoreload
%autoreload 2

In [3]:
# Path to the folder with the scans and the corresponding predictions
predictions_files = sorted(glob(os.path.join(LIVER_LESIONS_DATASET, "*support_analysis_10.nii.gz")))
pattern = re.compile(r".*\d+_support_analysis_10.nii.gz")
predictions_files = [path for path in predictions_files if pattern.search(path)]
scans = [path.replace("_support_analysis_10.nii.gz", "_scan.nii.gz") for path in predictions_files]
liver_masks = [path.replace("_support_analysis_10.nii.gz", "_liver.nii.gz") for path in predictions_files]

In [4]:
# Load the scans and the predictions
scans = [nib.load(scan).get_fdata() for scan in scans]
predictions = [nib.load(prediction).get_fdata() for prediction in predictions_files]

In [5]:
liver_masks = [nib.load(liver_mask).get_fdata().astype("float64") for liver_mask in liver_masks]

In [6]:
# Get only predictions within the liver
predictions = [prediction * liver_mask for prediction, liver_mask in zip(predictions, liver_masks)]
# postprocess the predictions
post_predictions = postprocess_predictions(predictions, save_postprocessed=True, predictions_affines=[nib.load(path).affine for path in predictions_files], predictions_filenames=predictions_files)

100%|██████████| 93/93 [01:58<00:00,  1.27s/it]


In [18]:
# Initialize the VirtualRichard
virtual_richard = VirtualRichard()

In [19]:
# get the GT masks
gt_masks = [nib.load(path.replace("_support_analysis_10.nii.gz", "_seg.nii.gz")).get_fdata() for path in predictions_files]
post_gt_masks = postprocess_predictions(gt_masks)

100%|██████████| 93/93 [01:22<00:00,  1.13it/s]


In [20]:
detection_metrics = virtual_richard.evaluate_detection(predictions, gt_masks)

In [49]:
segmentation_metrics = virtual_richard.evaluate_segmentation(predictions, gt_masks)

In [22]:
TP_score = pd.DataFrame(detection_metrics[0])
FP_score = pd.DataFrame(detection_metrics[1])
FN_score = pd.DataFrame(detection_metrics[2])

In [23]:
TP_score.describe()

,0
count,93.000000
mean,0.511330
std,0.241411
min,0.000000
25%,0.357143
50%,0.500000
75%,0.666667
max,1.000000


In [24]:
FP_score.describe()

,0
count,93.000000
mean,0.963842
std,0.042767
min,0.781250
25%,0.951735
50%,0.980341
75%,0.990689
max,1.000000


In [25]:
FN_score.describe()

,0
count,93.000000
mean,0.488670
std,0.241411
min,0.000000
25%,0.333333
50%,0.500000
75%,0.642857
max,1.000000


In [46]:
countour_score = pd.DataFrame(np.concatenate(segmentation_metrics[0]))
rvd_score = pd.DataFrame(segmentation_metrics[1])

In [47]:
countour_score.describe()

,0
count,1066.000000
mean,0.883613
std,0.259977
min,0.003589
25%,0.990892
50%,1.000000
75%,1.000000
max,1.000000


In [28]:
rvd_score.describe()

,0
count,93.000000
mean,704.916215
std,5587.123273
min,-0.916918
25%,-0.241364
50%,0.861307
75%,7.209052
max,53502.000000
